# 健身动作模型 Kaggle 独立跑分

加载仓库里的 MM-Fit checkpoint，在 `test` split 上重新推理。

预期：accuracy ≈ 0.9996，macro-F1 ≈ 0.9997。


In [ ]:
import os, sys, json, subprocess, urllib.request, zipfile
from pathlib import Path

REPO = Path("/kaggle/working/repo")
if not REPO.exists():
    subprocess.run(["git", "clone", "https://github.com/smileyan369/fitness-action-kaggle.git", str(REPO)], check=True)
os.chdir(REPO)
print("cwd:", os.getcwd())
print(subprocess.run([sys.executable, "-c", "import torch; print(torch.__version__, torch.cuda.is_available())"], capture_output=True, text=True).stdout)


In [ ]:
DATA = Path("/kaggle/input/fitness-mmfit/mmfit")
if not DATA.exists():
    zpath = Path("/kaggle/working/mmfit.zip")
    url = "https://github.com/smileyan369/fitness-action-kaggle/releases/download/v1.0-data/kaggle_mmfit_processed.zip"
    print("mount not found; downloading data...")
    urllib.request.urlretrieve(url, zpath)
    with zipfile.ZipFile(zpath) as z:
        z.extractall("/kaggle/working/mmfit_data")
    DATA = Path("/kaggle/working/mmfit_data/mmfit")
CKPT = REPO / "cloud_results" / "runs" / "mmfit_bilstm_s42" / "best.pt"
OUT = Path("/kaggle/working/eval_mmfit")
assert DATA.exists(), DATA
assert CKPT.exists(), CKPT
subprocess.run([sys.executable, "-m", "src.evaluate", "--checkpoint", str(CKPT), "--data", str(DATA), "--split", "test", "--out", str(OUT)], check=True)


In [ ]:
import pandas as pd
m = json.loads((OUT / "metrics.json").read_text())
print("accuracy:", m["accuracy"])
print("macro_f1:", m["macro_f1"])
display(pd.DataFrame(m["per_class"]).T)
display(pd.read_csv(OUT / "confusion_matrix.csv", index_col=0))


通过标准：accuracy 0.995–1.000，macro-F1 0.995–1.000。

注意：这是 MM-Fit 已切好的动作组 test split，不是连续视频实时识别。
